# ResNet-18 physical PYNQ-Z1 demo
This is the canonical human demo. Open it from a deployed release on the PYNQ-Z1, select the board's PYNQ Python kernel, and run the cells in order. The final cell must show physical NPU evidence; a host-only backend cannot pass.

In [ ]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

if not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook on the PYNQ-Z1, not Windows')

import pynq  # proves that the selected kernel provides the board runtime

release_root = next(
    (candidate for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (candidate / 'deployment.json').is_file()
     and (candidate / 'examples/resnet18/run_on_board.py').is_file()),
    None,
)
if release_root is None:
    raise RuntimeError('Open the notebook from a release created by deploy_release.ps1')
deployment = json.loads((release_root / 'deployment.json').read_text(encoding='utf-8'))
print(f'Release: {release_root}')
print(f"PYNQ: {getattr(pynq, '__version__', 'unknown')}")

## Inspect what will be tested
Confirm the deployed and artifact commits below before starting the long physical run. A mismatch is development evidence only.

In [ ]:
assert deployment['magic'] == 'NPU_RESNET18_DEPLOYMENT'
print(json.dumps(deployment, indent=2, sort_keys=True))

## Run physical acceptance
This cell loads the real overlay and model, executes the NPU, and writes a new evidence file. It can take a long time.

In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
evidence_path = release_root / f'notebook-evidence-{stamp}.json'
command = [
    sys.executable,
    'examples/resnet18/run_on_board.py',
    '--artifact-dir', 'build/vivado/npu_matrix/artifacts',
    '--expected-source-commit', deployment['artifact_source_commit'],
    '--deployed-source-commit', deployment['deployed_source_commit'],
    '--evidence', str(evidence_path),
]
if deployment['allow_source_mismatch']:
    command.append('--allow-source-mismatch')
environment = os.environ.copy()
environment['XILINX_XRT'] = '/usr'
completed = subprocess.run(
    command, cwd=release_root, env=environment, text=True, capture_output=True
)
print(completed.stdout, end='')
if completed.stderr:
    print(completed.stderr, file=sys.stderr, end='')
completed.check_returncode()

## Human acceptance
Review the evidence rather than trusting only the PASS text. The runtime must report physical jobs and the captured hashes must match host acceptance.

In [ ]:
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
assert evidence['result'] == 'pass'
assert evidence['runtime']['physical_jobs'] > 0
assert evidence['evidence_type'] in {
    'physical-pynq-z1', 'physical-pynq-z1-development'
}
print(f"PASS [{evidence['evidence_type']}]: human-reviewed notebook demo")
evidence